# Autonomous Driving VLM Representation Dynamics & Language Prior Analysis

Hệ thống phân tích và chẩn đoán động học biểu diễn của mô hình **Question-Guided Token Router** (và baseline GPA) cho xe tự hành (nuScenes / DriveLM) dựa trên phương pháp luận phân kỳ biểu diễn đối sánh ($D_l$, VIP, TVI) từ bài báo ICLR 2026 (*Understanding Language Prior of LVLMs by Contrasting Chain-of-Embedding*).

### Tính năng chính:
1. **Tự động nhận diện GPU:** Tận dụng tối đa Kaggle GPU (Tesla T4 x2 hoặc P100).
2. **Tự động tìm kiếm Checkpoint & Feature Cache:** Quét thông minh từ `/kaggle/input/`.
3. **Thực thi 1-Click:** Tự động thực hiện suy luận kép ($VL$ vs $L$), tính toán $D_l$, phát hiện tầng VIP, tính TVI và vẽ biểu đồ.
4. **Hiển thị trực tiếp:** Hiển thị trực quan biểu đồ vector PDF và PNG ngay trong notebook.
5. **Đóng gói tải về:** Tự nén file zip `analysis_results_bundle.zip` để tải về máy với 1 click.

In [ ]:
# 1. Khởi tạo môi trường & Cài đặt thư viện bổ trợ
import os, sys, platform
from pathlib import Path

!pip install -q pyyaml scipy matplotlib transformers tqdm

# Kiểm tra Kaggle Secrets (nạp HF_TOKEN nếu có)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("[Auth] HF_TOKEN loaded successfully from Kaggle Secrets!")
except Exception:
    print("[Auth] No Kaggle Secrets HF_TOKEN found. Using standard public rate limits.")

In [ ]:
# 2. Kiểm tra thông tin GPU & Phần cứng Kaggle
import torch

print(f"Python Version: {platform.python_version()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

!nvidia-smi

In [ ]:
# 3. Chạy Pipeline Phân Tích 1-Click (scripts/kaggle_run_analysis.py)
# Chú ý: Script sẽ tự động tìm kiếm checkpoint và cache trong /kaggle/input/
# Bạn có thể truyền trực tiếp --checkpoint và --cache_dir nếu muốn chỉ định đường dẫn cụ thể.

!python scripts/kaggle_run_analysis.py \
    --config configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49.yaml \
    --output_dir /kaggle/working/outputs/ad_analysis \
    --model_name "Question-Guided Router (K=49) + RepViT-T5" \
    --batch_size 16 \
    --device auto

In [ ]:
# 4. Trực Quan Hóa Kết Quả & Biểu Đồ Trực Tiếp Trong Notebook
from IPython.display import Image, display

output_dir = Path("/kaggle/working/outputs/ad_analysis")
curve_img = output_dir / "divergence_curve.png"
multi_img = output_dir / "multi_panel_analysis.png"

if curve_img.exists():
    print("=" * 60)
    print("BIỂU ĐỒ PHÂN KỲ TẦNG D_l (D_T vs D_VT) & TẦNG VIP")
    print("=" * 60)
    display(Image(filename=str(curve_img), width=900))

if multi_img.exists():
    print("=" * 60)
    print("BIỂU ĐỒ 2 PANEL: PHÂN KỲ TẦNG & TRỌNG SỐ CHÚ Ý THỊ GIÁC")
    print("=" * 60)
    display(Image(filename=str(multi_img), width=1050))

In [ ]:
# 5. Xem Chi Tiết Chỉ Số TVI & Tải Về Gói Kết Quả
import json

metrics_file = output_dir / "divergence_metrics.json"
if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
    print("=" * 60)
    print("TỔNG HỢP CHỈ SỐ KHOA HỌC (METRICS SUMMARY):")
    print("=" * 60)
    print(f"Tổng số mẫu đánh giá:      {metrics['total_samples']}")
    print(f"Tỷ lệ Language Prior (D_T): {metrics['language_prior_ratio_pct']:.1f}%")
    print(f"Tầng VIP sử dụng:           Layer {metrics['vip_layer_used']} (Auto: Layer {metrics['vip_layer_auto']})")
    print(f"Điểm TVI (D_T):             {metrics['tvi']['dt_mean']:.4f}")
    print(f"Điểm TVI (D_VT):            {metrics['tvi']['dvt_mean']:.4f}")
    print(f"Khoảng cách TVI Gap:        {metrics['tvi']['tvi_gap']:+.4f}")
    print("=" * 60)

zip_file = output_dir / "analysis_results_bundle.zip"
if zip_file.exists():
    print(f"\nĐã đóng gói hoàn chỉnh file: {zip_file} ({zip_file.stat().st_size} bytes)")
    print("Bạn có thể click chuột vào file này ở bảng 'Data > Output' bên phải để tải về máy tính!")